In [6]:
import numpy as np
from langchain_core.documents import Document
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))
from src.embeddings.embedding_manager import EmbeddingManager
from src.vector_db.vector_store import VectorStore


In [ ]:
class Retriever:

    def __init__(self, embedding_manager:EmbeddingManager, vector_store:VectorStore ):
        
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrive( self, query:str, top_k: int=5, score_threshold: float=0.0)-> list[dict]:

        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.query(
                query_embedding=query_embedding.tolist(),top_k=top_k
            )

            # print(results)
            retrived_docs=[]
            
            if results['documents'] and results['metadatas']and results['distances'] and results['documents'][0]:
            
                ids=results['ids'][0]
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score=1-distance

                    if similarity_score >=score_threshold :
                    # if True:
                        retrived_docs.append({
                            "id": doc_id,
                            "content":document,
                            "metadata":metadata,
                            'similarity_score':similarity_score,
                            "distance":distance,
                            'rank':i+1
                        })

                print(f" number of retrived docs {len(retrived_docs)}")
            else:
                print(f"documents are not found")
            return retrived_docs

        except Exception as e:
            print( f"there is error in the retrival {e}")
            return []
        


    




